# Lab 4 — Tool-Use Patterns (§19.2.4)

| Pattern | คำอธิบาย | ตัวอย่างใน lab |
|---|---|---|
| Single-turn | tool call เดียว, LLM 2 calls | calculator |
| Multi-tool | หลาย tool พร้อมกันใน response เดียว | morning brief |
| Sequential | output ของ tool หนึ่งเป็น input ของอีกตัว | search → fetch |
| Nested | tool call เรียก agent อีกตัว | math sub-agent |
| Fallback | tool หลักล้มเหลว → ลองตัวสำรอง | primary → backup → cache |

> อ้างอิง: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §19.2.4 — CC BY-SA 4.0

> ✏️ **Starter notebook — ฉบับให้ผู้เรียนลงมือเขียน**
>
> โค้ดส่วนใหญ่เตรียมไว้ให้แล้ว เขียนเพิ่มเฉพาะ **4 จุดที่มี `✏️ TODO`** (แทนที่ `___` หรือ `...` ด้วยโค้ดของคุณ) ทุกจุดบอกเวลาในวิดีโอที่อธิบายโค้ดตรงนั้น (▶ m:ss) — ดูคลิปก่อน แล้วลองเขียนเองโดยไม่เปิดเฉลย
>
> รันทุก cell ตามลำดับจากบนลงล่าง ถ้า cell ไหนยังเติมไม่ครบจะเจอ `NameError` หรือ `TypeError` เป็นเรื่องปกติ
>
> ติดตรงไหน เปิดเฉลยใน **`ep05-lab4-tool-use.ipynb`** (โค้ดชุดเดียวกับในวิดีโอ)

## 0. เตรียมเครื่อง

ต้องมี Python 3.10+ (หรือ Colab) และ **OpenRouter API key** — วิธีตั้งค่าอยู่ใน Lab 1

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

In [ ]:
# ---- จาก Lab 1: client + tools + react_agent (+ ตัวนับ LLM calls และจับเวลา tools) ----
import os, json, time
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

def load_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        from dotenv import load_dotenv
        load_dotenv()
        return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"provider": {"require_parameters": True}, "reasoning": {"enabled": False}}  # lab นี้ไม่ใช้ Thought
CALLS = {"n": 0}

def ask_llm(messages, tools=None, system=None):
    if system:
        messages = [{"role": "system", "content": system}] + messages
    kwargs = dict(model=MODEL, messages=messages, extra_body=EXTRA)
    if tools:
        kwargs["tools"] = tools
    CALLS["n"] += 1
    return client.chat.completions.create(**kwargs).choices[0].message

TOOLS = {}

def tool(description, properties):
    def wrap(fn):
        TOOLS[fn.__name__] = {"fn": fn, "schema": {"type": "function", "function": {
            "name": fn.__name__, "description": description,
            "parameters": {"type": "object", "properties": properties,
                           "required": list(properties), "additionalProperties": False}}}}
        return fn
    return wrap

def schemas(names):
    return [TOOLS[n]["schema"] for n in names]

def run_tool(call):
    try:
        out = TOOLS[call.function.name]["fn"](**json.loads(call.function.arguments))
    except Exception as e:
        out = f"Error: {e}"
    return {"role": "tool", "tool_call_id": call.id, "content": str(out)}

def react_agent(task, tool_names, system=None, max_iterations=8, indent=""):
    messages = [{"role": "user", "content": task}]
    for i in range(1, max_iterations + 1):
        msg = ask_llm(messages, tools=schemas(tool_names), system=system)
        calls = msg.tool_calls or []
        if not calls:
            return msg.content
        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": [c.model_dump() for c in calls]})
        for c in calls:
            print(f"{indent}🔧 [{i}] {c.function.name}({c.function.arguments})")
        t0 = time.time()
        with ThreadPoolExecutor() as pool:
            results = list(pool.map(run_tool, calls))
        if len(calls) > 1:
            print(f"{indent}⏱️ {len(calls)} tools ran in parallel: {time.time() - t0:.1f}s")
        for r in results:
            print(f"{indent}👀 [{i}] {r['content'][:120]}")
        messages.extend(results)
    return "⚠️ Stopped: reached max_iterations"

## 1. Single-turn — tool เดียว, LLM 2 calls

In [ ]:
# ✏️ TODO 1 (▶ วิดีโอ 2:05): ป้องกัน eval: ยอมให้มีเฉพาะตัวเลข และ + - * / ( ) . กับช่องว่าง
@tool("Evaluate an arithmetic expression, e.g. '(3+4)*2'.", {"expression": {"type": "string"}})
def calculator(expression):
    if not ___:
        return "Error: only numbers and + - * / ( ) are allowed"
    return str(eval(expression))

In [ ]:
before = CALLS["n"]
print(react_agent("What is (17 * 23) + 9? Use the calculator.", ["calculator"]))
print("LLM calls:", CALLS["n"] - before)

## 2. Multi-tool (parallel)

tool ต้อง **อิสระต่อกัน** → harness รันพร้อมกัน → **latency = max(...)** แทนผลรวม

In [ ]:
def slow(value):
    time.sleep(1.5)
    return value

@tool("Get the latest stock price for a ticker.", {"ticker": {"type": "string"}})
def get_stock_price(ticker):
    return slow(json.dumps({"ticker": ticker, "price_usd": 187.4}))

@tool("Get today's weather for a city.", {"city": {"type": "string"}})
def get_weather(city):
    return slow(json.dumps({"city": city, "forecast": "rain, 29°C"}, ensure_ascii=False))

@tool("Get calendar events for a date (YYYY-MM-DD).", {"date": {"type": "string"}})
def get_calendar(date):
    return slow(json.dumps([{"time": "14:00", "event": "Thesis meeting"}]))

In [ ]:
import datetime
today = datetime.date.today().isoformat()
print(react_agent(f"Today is {today}. Give me a morning brief: AAPL stock price, Bangkok weather, "
                  "and my calendar for today. Fetch all three at once.",
                  ["get_stock_price", "get_weather", "get_calendar"]))

## 3. Sequential (pipeline)

output ของ tool หนึ่งเป็น input ของ tool ถัดไป — model ตัดสินใจ tool ถัดไปจากผลก่อนหน้า

In [ ]:
DOCS = {  # ข้อมูลจำลอง
    "https://docs.example/patchtst": "PatchTST splits each series into patches of 16 steps "
                                     "and models each channel independently.",
    "https://docs.example/itransformer": "iTransformer applies attention across variates instead of time steps.",
}

# ✏️ TODO 2 (▶ วิดีโอ 4:55): fetch_page: คืนเนื้อหาจาก DOCS ตาม url ถ้าไม่มี url นั้นให้คืนข้อความ Error แทนการ crash
@tool("Search the docs. Returns a JSON list of {title, url}.", {"query": {"type": "string"}})
def search_docs(query):
    return json.dumps([{"title": u.rsplit("/", 1)[1], "url": u} for u in DOCS])

@tool("Fetch a page by url and return its text.", {"url": {"type": "string"}})
def fetch_page(url):
    return ___

In [ ]:
print(react_agent("How long is each patch in PatchTST? Search the docs, then fetch the right page.",
                  ["search_docs", "fetch_page"]))

## 4. Nested (agent-as-tool)

parent agent เห็น sub-agent เป็นแค่ tool ตัวหนึ่ง (black box) แต่ข้างในคือ agent อีกตัวที่มี prompt และ tool ของตัวเอง

In [ ]:
# ✏️ TODO 3 (▶ วิดีโอ 6:20): agent-as-tool: tool นี้ภายในคือ react_agent อีกตัวที่มี calculator เป็น tool
@tool("Delegate a math word problem to a specialist math agent. Returns the worked answer.",
      {"problem": {"type": "string"}})
def ask_math_agent(problem):
    print("      ↳ 🤖 math sub-agent started")
    return ___(problem, ___,
               system="You are a careful math agent. Always use the calculator for arithmetic.",
               indent="      ")

In [ ]:
print(react_agent("A class has 3 sections of 42 students. Each student needs 2 notebooks "
                  "costing 18.5 baht each. How much do all notebooks cost? Delegate the math.",
                  ["ask_math_agent"]))

## 5. Fallback (graceful degradation)

harness ลอง tool ตามลำดับความสำคัญ — **model ไม่ต้องรู้เรื่อง fallback เลย** (หลัก *Plan for failure*, §19.3)

In [ ]:
def primary_search(q): raise TimeoutError("primary API timed out")
def backup_search(q):  raise RuntimeError("429 rate limited")
def cached_search(q):  return json.dumps([{"title": "PatchTST", "source": "cache (2 days old)"}])

# ✏️ TODO 4 (▶ วิดีโอ 7:53): ลองแหล่งค้นหาทีละตัวตามลำดับ ตัวไหนสำเร็จคืนผลทันที ถ้าล้มหมดให้คืนข้อความบอก model อย่างสุภาพ
@tool("Search the web for a topic.", {"query": {"type": "string"}})
def web_search(query):
    for name, fn in [("primary", primary_search), ("backup", backup_search), ("cache", cached_search)]:
        try:
            result = ___
            print(f"      ✅ {name} succeeded")
            return result
        except Exception as e:
            print(f"      ❌ {name} failed: {e}")
    return ___

In [ ]:
print(react_agent("Search the web for a patch-based time-series transformer and name one.", ["web_search"]))

## สรุป

| Pattern | สิ่งที่เห็นใน lab |
|---|---|
| Single-turn | LLM calls = 2 |
| Multi-tool | 3 tools ใช้ ~1.5s ไม่ใช่ 4.5s |
| Sequential | search_docs → fetch_page |
| Nested | sub-agent ที่มี prompt + tool ของตัวเอง |
| Fallback | primary ❌ → backup ❌ → cache ✅ |

**ลองทำต่อ:** สร้าง tool ที่ต้องใช้ผลของอีก tool แล้วขอให้ model "fetch all at once" — จะเกิดอะไรขึ้น?